# Comparative Reliability Performance of N-Body Simulation under Hamiltonian Framework Applying Barnes-Hut Trees
Within the realms of modern orbital mechanics application of Barnes-Hut trees is a common practice for large N-Body simulations. The project describes the creation of a C++ and R tool to simulate the problem assessing reliability of the Barnes-Hut trees for cosmological applications via application of symplectic and non-symplectic integration schemes.

## About The Notebook
This notebook is a simulation prototype, that shows fast results and validation of said results using Python.
It will be kept in a dated entry order expanding each individual feature for a later transfer to C++ and R framework.
It will also comply with the project's gantt chart of progress.

Please bear in mind that the function and variable names follow a certain pattern:
- Integrators and their daughter functions: `operation_integrator_order`
- Mathematical equations take their literal mathematical names, where numerator is a numerator, product is a product, etc.

## Setting the Environment and Creating Velocity Verlet 4th Order (28/10/26)

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import random as rnd

# Randomisation Seed
seed = 256
rnd.seed(seed)

# Constants
yoshida_x0 = -(2**(1/3))/(2-(2**(1/3)))
yoshida_x1 = (1/(2-(2**(1/3))))

In [ ]:
"""
Creating helper functions to help with force evaluation later.
"""

# Find Euclidian distance between two bodies
# obj_a – object i with accessible properties
# obj_b - object j with accessible properties
def euclidian_distance(obj_a, obj_b):
    return np.linalg.norm(obj_a.r - obj_b.r)

# Find initial velocity
# obj – individual object with accessible properties
# G – constant of gravitational attraction
# distance – previously found Euclidian distance
def vel_init(obj, G, distance):
    return rnd.random() * np.sqrt((G*obj.m) / distance)  # random variable in the beginning is used to create random orbit eccentricities (no ejection)

# Get a perpendicular vector
def get_perpendicular(v):
    x, y, z = v
    if x != 0 or y != 0:
        perp = np.array([-y, x, 0.0], dtype=float)
    else:
        perp = np.array([0.0, 1.0, 0.0], dtype=float)

    return perp

# Find velocity direction
# Essentially convert scalar velocity into vector velocity
# obj_a – object i with accessible properties
# obj_b - object j with accessible properties
# distance – previously found Euclidian distance
# v – scalar velocity
def vel_direction(obj_a, obj_b, distance, v):
    radius_vector = obj_b.r - obj_a.r
    perp_vector = get_perpendicular(radius_vector)
    unit_perp_vector = perp_vector / np.linalg.norm(perp_vector)    # normalise by its own length

    return v * unit_perp_vector


# Compute Gravitational Acceleration of All Bodies acting on i
# G – the constant of gravitational attraction
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
def acc(G, bod_obj, e):
    N = len(bod_obj)    
    accelerations = [np.zeros(3) for _ in range(N)]     # initialise an array to hold acceleration vectors for all bodies

    # Sigma operation
    # Loop over every body i
    for i in range(N):
        a_i = np.zeros(3)   # net acceleration vector on body i

        # Sum gravitational pull from every other body j
        for j in range(N):
            if i == j:
                continue    # idiot-proofing self-interaction

            # Compute displacement from i to j
            r_ij = bod_obj[i].r - bod_obj[j].r

            # Scalar distance squared
            dist_sq = np.sum(r_ij**2)

            # Denominator with softening factor
            denominator = (dist_sq + e**2)**1.5

            # Add the j-body contribution
            a_i += bod_obj[j].m * (r_ij / denominator)

        # Apply -G and store the net acceleration for body i
        accelerations[i] = -G * a_i

    return accelerations


In [ ]:
"""
Creating a Velocity Verlet 4th Order Script.
It consists of the acceleration, displacement and velocity computations.

The detailed description of the algorithm will be provided later down below.
"""

# Compute Displacement of an individual body
# obj – individual object with accessible properties
# t – current timestep
def displ_vv_4(obj, t):
    return obj.r + obj.v*t + (0.5*obj.a*(t**2))

# Compute Velocity of an individual body
# obj – individual object with accessible properties
# t – current timestep
# a_old – old acceleration of the body
def vel_vv_4(obj, t, a_old):
    return obj.v + 0.5*(a_old+obj.a)*t

# Run Velocity Verlet (4th order, Yoshida composition of three 2nd order steps)
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_vv_4(bod_obj, G, e, dt, n_steps):
    weights = (yoshida_x1, yoshida_x0, yoshida_x1)    # sub-step sizes as fractions of dt (they sum to 1)

    # Initial accelerations of all bodies
    for obj, a in zip(bod_obj, acc(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for step in range(n_steps):
        # Yoshida sub-steps: each one is a full Velocity Verlet step of size w*dt
        for w in weights:
            h = w * dt

            a_old = [obj.a for obj in bod_obj]    # accelerations before the move

            # Move ALL bodies first (new displacement)
            for obj in bod_obj:
                obj.r = displ_vv_4(obj, h)

            # Re-evaluate the forces once for the whole system (new acceleration)
            for obj, a in zip(bod_obj, acc(G, bod_obj, e)):
                obj.a = a

            # Update ALL velocities with the average of old and new acceleration
            for obj, a_prev in zip(bod_obj, a_old):
                obj.v = vel_vv_4(obj, h, a_prev)


## Velocity Verlet 4th Order Explanation
In order to solve the N-body problem numerically, several quantities have to be evaluated: the Euclidean distance between pairs of bodies, an initial velocity vector obtained from a randomised scalar speed, and the gravitational acceleration, which is then propagated in time by the integrator. The initial speed is scaled by a random factor in order to produce a variety of orbital eccentricities for experimental purposes.

**Notation.** Bodies are indexed by $i, j = 1, \dots, N$. Vectors carry a hat in the convention used throughout this notebook: $\hat{r}_i$ is the displacement, $\hat{v}_i$ the velocity and $\hat{a}_i$ the acceleration of body $i$, while $m_i$ is its mass. $G$ is the gravitational constant, $\delta{t}$ is the timestep, $\epsilon$ is the softening length (`e` in the code) and $\xi$ is the random eccentricity factor.

### Euclidean Distance in 3D
The separation of two bodies is the length of the difference of their displacement vectors (`euclidian_distance`):

$$r_{ij}=\lvert\hat{r}_i-\hat{r}_j\rvert=\sqrt{(x_i-x_j)^2 + (y_i - y_j)^2 + (z_i-z_j)^2}\tag{1}$$

### Initial Speed
The speed of a circular orbit at distance $r_{ij}$ around a body of mass $M$ is $v_c=\sqrt{GM/r_{ij}}$, where $M$ is the mass of the attracting body. The initial speed is a random fraction of it (`vel_init`):

$$v_0=\xi\sqrt{\frac{GM}{r_{ij}}},\qquad \xi\sim\mathcal{U}[0,1)\tag{2}$$

The velocity is launched perpendicular to the radius vector, so the launch point is an apsis of the resulting orbit. For $\xi<1$ the body starts at apocentre and follows an ellipse of eccentricity $e_{\mathrm{orb}}=1-\xi^2$, while $\xi\to1$ gives a circular orbit. Since the escape speed $\sqrt{2}\,v_c$ is never reached, no body is ejected at the start of the simulation.

### Initial Velocity Direction
The scalar speed is converted into a vector by choosing a direction perpendicular to the radius vector $\Delta\hat{r}=\hat{r}_j-\hat{r}_i=(\Delta x,\Delta y,\Delta z)$ (`get_perpendicular` and `vel_direction`):

$$\hat{p}=(-\Delta y,\ \Delta x,\ 0),\qquad \hat{n}=\frac{\hat{p}}{\lvert\hat{p}\rvert},\qquad \hat{v}_0=v_0\,\hat{n}\tag{3}$$

The vector $\hat{p}$ is perpendicular to $\Delta\hat{r}$ because $\hat{p}\cdot\Delta\hat{r}=-\Delta x\Delta y+\Delta y\Delta x=0$. If $\Delta x=\Delta y=0$ the construction degenerates, and $\hat{p}=(0,1,0)$ is used instead. As $\hat{n}$ lies in the $xy$-plane, the initial orbits are planar.

### Gravitational Acceleration
Newton's law of gravitation, summed over all other bodies, gives the acceleration of body $i$ (`acc`). A softening length $\epsilon$ is added to the denominator (Plummer softening) so that the force stays finite when two bodies approach each other:

$$\hat{a}_i=-G\sum_{j\neq i}\frac{m_j\,(\hat{r}_i-\hat{r}_j)}{\left(\lvert\hat{r}_i-\hat{r}_j\rvert^2+\epsilon^2\right)^{3/2}}\tag{4}$$

This direct summation requires $\mathcal{O}(N^2)$ pair evaluations per call, which is precisely the cost that the Barnes-Hut tree reduces to $\mathcal{O}(N\log N)$ in the later C++ implementation.

### Velocity Verlet (Second Order)
Since the algorithm is a 4th-order Velocity Verlet, it integrates with a different pattern to the regular Verlet.
Regular 2nd-order Velocity Verlet uses the evaluation in a following pattern:
- Set up initial conditions
- Evaluate displacement
- Evaluate acceleration
- Evaluate velocity

I coined a name for this method, RAV, due to $\hat{r}$ being displacement, $\hat{a}$ acceleration and $\hat{v}$ velocity. Velocity Verlet can be expressed via the following equations, where a single step is denoted $S_2(\delta{t})$:

**Displacement** (`displ_vv_4`)

$$\hat{r}(t+\delta{t}) = \hat{r}(t)+\hat{v}(t)\,\delta{t}+\frac{\hat{a}(t)\,\delta{t}^2}{2}\tag{5}$$

**Velocity** (`vel_vv_4`)

$$\hat{v}(t+\delta{t})=\hat{v}(t)+\frac{\hat{a}(t)+\hat{a}(t+\delta{t})}{2}\,\delta{t}\tag{6}$$

The new acceleration $\hat{a}(t+\delta{t})$ in Eq. (6) is evaluated from the new displacements of Eq. (5), which is why the order of the three operations cannot be changed. The scheme is time-reversible and symplectic, but only second-order accurate: its global error scales as $\mathcal{O}(\delta{t}^2)$.

### Yoshida Composition to 4th Order
Yoshida's method raises the order by composing three second-order steps of different sizes:

$$S_4(\delta{t})=S_2(w_1\delta{t})\circ S_2(w_0\delta{t})\circ S_2(w_1\delta{t})\tag{7}$$

$$w_1=\frac{1}{2-2^{1/3}}\approx 1.3512,\qquad w_0=-\frac{2^{1/3}}{2-2^{1/3}}\approx -1.7024\tag{8}$$

In the code these are `yoshida_x1` and `yoshida_x0`. The weights are fixed by two conditions: the sub-steps must add up to one full step, and the leading error term of $S_2$ must cancel,

$$2w_1+w_0=1,\qquad 2w_1^3+w_0^3=0\tag{9}$$

so that the local error falls from $\mathcal{O}(\delta{t}^3)$ to $\mathcal{O}(\delta{t}^5)$ and the global error to $\mathcal{O}(\delta{t}^4)$. The middle weight $w_0$ is negative, so the middle sub-step runs backwards in time, but the three sub-steps together advance the system by exactly $\delta{t}$, and the composition remains symplectic. The price is three force evaluations per timestep instead of one.

### One Integration Step in the Code
For each of the three sub-steps of size $h=w\,\delta{t}$ with $w\in\{w_1, w_0, w_1\}$ (`run_vv_4`):
1. Store the current accelerations $\hat{a}_{\mathrm{old}}$ of all bodies.
2. Move all bodies using Eq. (5) with $\delta{t}\to h$.
3. Re-evaluate the accelerations of all bodies with Eq. (4) at the new positions.
4. Update all velocities using Eq. (6) with $\delta{t}\to h$ and $\hat{a}_{\mathrm{old}}$.

All bodies are moved before any acceleration is recomputed, so that every force is evaluated for the same instant.

In [ ]:
"""
Experiment 1: the three-body figure-eight choreography (28/10/26).

Purpose:
    The figure-eight is a periodic solution of the planar three-body problem in which
    three equal masses chase one another along a single closed curve shaped like the
    number eight. It was found numerically by Moore (1993) and its existence was proven
    by Chenciner & Montgomery (2000). Because the exact answer is known (the bodies must
    return to their starting positions after every period T), it is an ideal first
    validation test for the integrator: any drift away from the starting positions after
    a whole number of periods is a direct measure of the numerical error.

Setup:
    - three equal masses chasing each other along a single figure-eight curve
    - initial conditions from Simo's high-precision values, in natural units G = m = 1
    - no softening: the bodies never come close, and softening would perturb the orbit
    - integrated with run_vv_4 (Velocity Verlet 4th order)

Time control:
    - the total simulated time is a whole number of orbital periods
    - dt is dictated by the number of timesteps: dt = t_total / n_steps
    - frames are computed separately: one trajectory point is stored every
      steps_per_frame = n_steps / n_frames steps
No energy tracking yet; the output is the trajectories plus the closure error after the run.
"""


# A minimal body: mass, displacement, velocity and acceleration.
# The integrator (run_vv_4) and the force routine (acc) only ever access the
# attributes m, r, v and a, so any object that carries these four attributes
# can be simulated. All vectors are stored as 3-component NumPy arrays, so that
# the same code works for planar (z = 0) and fully three-dimensional problems.
class Body:
    def __init__(self, m, r, v):
        self.m = m      # mass (scalar)
        self.r = np.array(r, dtype=float)       # displacement vector (x, y, z)
        self.v = np.array(v, dtype=float)       # velocity vector (vx, vy, vz)
        self.a = np.zeros(3)        # acceleration vector, filled in by run_vv_4 on its first call


# Experiment parameters (natural units: G = 1, m = 1)
# The published figure-eight initial conditions are dimensionless: they assume G = 1
# and unit masses. Working in SI units would require rescaling every quantity, so the
# experiment is run directly in these natural units. Lengths, times and velocities
# printed below are therefore dimensionless as well.
G_nat = 1.0     # gravitational constant in natural units

# Plummer softening (Eq. 4) is switched off. The closest approach between any two
# bodies on the figure-eight is about 0.7 length units, so there is no risk of a
# division by zero, and any non-zero softening would alter the force law and hence
# destroy the periodicity that this experiment relies on.
e_soft = 0.0    # softening length (switched off, see above)

# Period of one full figure-eight orbit in natural units. After exactly one period
# every body returns to its own starting position. The value is only known to a
# finite number of digits, which sets a lower floor on the closure error below.
T_period = 6.3259093     # period of the figure-eight orbit

# The three numbers below fully control the time discretisation:
#   n_periods -> how long the simulation runs (t_total = n_periods * T_period)
#   n_steps   -> how finely that time is divided (this alone dictates dt)
#   n_frames  -> how many snapshots are stored for plotting (independent of dt)
# Increasing n_steps makes dt smaller and the result more accurate (the global error
# of the 4th-order scheme falls as dt^4) at a proportionally higher cost, while
# n_frames only changes the resolution of the plots and the memory used.
n_periods = 3   # number of full orbits to simulate
n_steps = 30000     # total number of timesteps (dictates dt)
n_frames = 1500     # number of stored trajectory points (independent of dt)

# Derived time quantities
t_total = n_periods * T_period      # total simulated time
dt = t_total / n_steps      # timestep implied by the chosen number of steps

# The integration is split into n_frames equal blocks of steps_per_frame steps each,
# and one snapshot is stored at the end of every block. For the blocks to add up to
# exactly n_steps, n_steps must be an integer multiple of n_frames; otherwise the run
# would stop short of (or overshoot) t_total and the closure test would be invalid.
assert n_steps % n_frames == 0, "n_steps must be divisible by n_frames"
steps_per_frame = n_steps // n_frames       # integration steps between two stored snapshots

# Initial conditions
# The configuration at t = 0 is symmetric:
#   - body 2 sits exactly at the crossing point (centre) of the eight
#   - bodies 1 and 3 sit at mirror-image positions, r3 = -r1
#   - bodies 1 and 3 share the same velocity, and body 2 moves with -2 times it
# These conditions give zero total momentum (m*v1 + m*v2 + m*v3 = 0), so the centre of
# mass stays fixed at the origin and no correction for drift is needed (unlike in a
# system with a randomly launched star). All z-components are zero, so the motion
# remains in the x-y plane for the entire run.
r1 = [0.97000436, -0.24308753, 0.0]     # displacement of body 1 (body 3 is its mirror image)
v2 = [-0.93240737, -0.86473146, 0.0]    # velocity of the central body 2
v13 = [-0.5 * v2[0], -0.5 * v2[1], 0.0]     # v1 = v3 = -v2 / 2, so the total momentum is zero

# The three bodies, each with unit mass
bodies = [
    Body(1.0, r1, v13),     # body 1: right-hand lobe
    Body(1.0, [0.0, 0.0, 0.0], v2),     # body 2: centre of the eight
    Body(1.0, [-r1[0], -r1[1], 0.0], v13),      # body 3: left-hand lobe
]

# Copies of the starting positions, kept for the closure test at the end of the run.
# .copy() is essential: without it r_start would refer to the same arrays that the
# integrator overwrites, and the closure error would always read zero.
r_start = [b.r.copy() for b in bodies]

# Summary of the time discretisation actually used
print(f"t_total = {t_total:.4f} ({n_periods} periods), n_steps = {n_steps}, dt = {dt:.3e}")
print(f"n_frames = {n_frames}, steps_per_frame = {steps_per_frame}")

# Simulation
# trajectories[k] is the list of stored displacements of body k. It starts with the
# initial position, so after the loop each list holds n_frames + 1 points (t = 0 plus
# one point per block). run_vv_4 advances the bodies in place by steps_per_frame steps
# of size dt; after each call the new positions are copied into the record.
# NB: run_vv_4 recomputes the accelerations at the start of every call. This costs one
# extra force evaluation per frame but does not change the result, because the
# accelerations depend only on the current positions.
trajectories = [[b.r.copy()] for b in bodies]
for _ in range(n_frames):
    run_vv_4(bodies, G_nat, e_soft, dt, steps_per_frame)
    for k, b in enumerate(bodies):
        trajectories[k].append(b.r.copy())

# Convert the nested lists into a single array for convenient slicing:
#   axis 0 -> body index k, axis 1 -> frame index, axis 2 -> coordinate (x, y, z)
trajectories = np.array(trajectories)       # shape: (N bodies, n_frames + 1, 3)

# Validation
# Closure check: after a whole number of periods every body should be back where it
# started. The distance |r(t_total) - r(0)| therefore measures the accumulated error.
# It contains two contributions: the integration error of the scheme (which shrinks as
# dt^4 when n_steps is increased) and the finite precision of T_period and of the
# initial conditions (which does not shrink with dt). Once the error stops decreasing
# when n_steps is raised, the second contribution has become dominant.
for k, b in enumerate(bodies):
    print(f"body {k + 1}: closure error |r(T) - r(0)| = {np.linalg.norm(b.r - r_start[k]):.3e}")

# Planarity check: with all z-components initially zero, the accelerations of Eq. (4)
# have no z-component either, so z must remain exactly zero for the whole run.
print(f"max |z| over the run = {np.abs(trajectories[:, :, 2]).max():.3e} (the orbit is planar)")

# Plots
colours = plt.cm.tab10(np.arange(3))        # one distinct colour per body

# All three bodies travel along the very same curve, so lines of equal width would hide
# one another. Drawing them with decreasing widths (body 1 widest, body 3 thinnest)
# keeps every trajectory visible as a coloured band inside the previous one.
line_widths = [3.0, 1.8, 0.8]
fig = plt.figure(figsize=(14, 6.5))

# 3D view
# The orbit is planar, so this view shows the eight lying flat at z = 0. It is kept in
# the experiment because later experiments (and the Barnes-Hut C++ code) are fully
# three-dimensional, and any spurious out-of-plane motion would be visible here.
ax3d = fig.add_subplot(1, 2, 1, projection="3d")
for k in range(len(bodies)):
    x, y, z = trajectories[k].T     # transpose (frames, 3) -> three coordinate arrays
    ax3d.plot(x, y, z, color=colours[k], lw=line_widths[k], label=f"Body {k + 1}")
    ax3d.scatter(x[0], y[0], z[0], color=colours[k], marker="x", s=30)     # starting position
    ax3d.scatter(x[-1], y[-1], z[-1], color=colours[k], s=30)       # final position

# Use the same limits on all three axes and a cubic box, so that lengths in different
# directions are directly comparable and the orbit is not visually distorted.
lim = 1.1 * np.abs(trajectories).max()      # 10% margin around the furthest point reached
ax3d.set_xlim(-lim, lim)
ax3d.set_ylim(-lim, lim)
ax3d.set_zlim(-lim, lim)
ax3d.set_box_aspect((1, 1, 1))
ax3d.set_xlabel("x")
ax3d.set_ylabel("y")
ax3d.set_zlabel("z")
ax3d.set_title("Figure-eight orbit in 3D")
ax3d.legend(fontsize=8, loc="upper left")

# Top-down projection (x-y plane)
# Since the motion is planar, this projection loses no information and shows the shape
# of the eight most clearly. Because the run covers a whole number of periods, the end
# markers (o) should lie on top of the start markers (x); a visible gap between them
# would indicate a large closure error.
ax2d = fig.add_subplot(1, 2, 2)
for k in range(len(bodies)):
    x, y, _ = trajectories[k].T     # the z-coordinate is not needed for the projection
    ax2d.plot(x, y, color=colours[k], lw=line_widths[k], label=f"Body {k + 1}")
    ax2d.scatter(x[0], y[0], color=colours[k], marker="x", s=40)        # starting position
    ax2d.scatter(x[-1], y[-1], color=colours[k], s=30)      # final position
ax2d.set_aspect("equal")        # equal scaling on x and y, so the eight is not stretched
ax2d.set_xlabel("x")
ax2d.set_ylabel("y")
ax2d.set_title("Projection onto the x-y plane (x = start, o = end)")
ax2d.grid(alpha=0.3)
ax2d.legend(fontsize=8)

fig.suptitle(f"Three-body figure-eight: {n_periods} periods, {n_steps} steps, "
             f"dt = {dt:.2e} (Velocity Verlet 4th order)")
plt.tight_layout()
plt.show()

In [ ]:
"""
Experiment 1 (animation): following the figure-eight in 3D, timestep by timestep (28/10/26).

This cell does not integrate anything; it replays the snapshots stored in `trajectories`
by the experiment cell above, so that cell must be run first. The animation is rendered
into an interactive HTML player (play/pause, step forwards/backwards, speed control and
a frame slider), which makes it possible to stop at any timestep and inspect the state.

What is shown in every frame:
    - a faint "ghost" of the full figure-eight curve, as a reference path
    - each body as a glowing marker, followed by a fading trail of its recent positions
    - the velocity vector of each body (an arrow pointing where it is about to go)
    - the centre of mass (a white cross), which must stay fixed at the origin
    - a read-out of the integration step, the simulated time, the orbital phase t/T and
      the speed of every body
The camera slowly orbits around the system so that the three-dimensional structure of
the scene (and the planarity of the orbit) can be appreciated.
"""

from matplotlib import animation
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from IPython.display import HTML

# Animation parameters
# The experiment stored n_frames + 1 snapshots. Rendering every one of them would make
# the HTML player very large and slow to build, so only every anim_stride-th snapshot
# becomes an animation frame. With n_frames = 1500 and anim_stride = 10, the player
# holds 151 frames, i.e. about 50 frames per orbital period.
anim_stride = 10        # stored snapshots skipped between two animation frames
trail_len = 150     # trail length in stored snapshots (150 of 500 per period = 0.3 T)
v_scale = 0.35      # length of the drawn velocity arrow per unit of speed
fps = 20        # playback speed of the player [frames per second]

# Indices (into the stored snapshots) of the frames that will actually be drawn
frame_ids = np.arange(0, trajectories.shape[1], anim_stride)

# Raise matplotlib's size limit for embedded animations (default 20 MB), otherwise
# longer runs or a smaller anim_stride would be silently truncated.
plt.rcParams["animation.embed_limit"] = 100     # [MB]

# Derived quantities for the read-out
# Time elapsed between two consecutive stored snapshots
dt_frame = steps_per_frame * dt

# Only positions were stored during the run, so the velocities are reconstructed here
# with central finite differences: v(t) ~ [r(t + dt_frame) - r(t - dt_frame)] / (2 dt_frame).
# np.gradient applies exactly this formula in the interior (and one-sided differences at
# the two ends). The error is of order dt_frame^2, which is far below what is visible
# on screen, so this is adequate for display (but NOT for energy diagnostics).
velocities = np.gradient(trajectories, dt_frame, axis=1)     # shape: (N bodies, n_frames + 1, 3)
speeds = np.linalg.norm(velocities, axis=2)     # shape: (N bodies, n_frames + 1)

# Centre of mass at every snapshot: r_cm = sum(m_k r_k) / sum(m_k).
# It is used as a live sanity check: with zero total momentum it must never move.
masses = np.array([b.m for b in bodies])
r_cm = np.einsum("k,kfc->fc", masses, trajectories) / masses.sum()      # shape: (n_frames + 1, 3)

# Figure and static elements
# A dark background makes the glowing markers and fading trails stand out.
bg = "#0b0e1a"
# dpi = 80 keeps every frame at 640 x 640 pixels, so the embedded player stays at a
# manageable size inside the notebook file (roughly 12 MB for 151 frames).
fig = plt.figure(figsize=(8, 8), dpi=80, facecolor=bg)
ax = fig.add_subplot(projection="3d", facecolor=bg)

# Style the 3D box: transparent panes with faint edges, light tick labels
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.set_pane_color((1, 1, 1, 0.02))
    axis._axinfo["grid"]["color"] = (1, 1, 1, 0.08)
    axis.label.set_color("0.7")
ax.tick_params(colors="0.5", labelsize=7)

# Same cubic box as in the static 3D plot, so the two views can be compared directly
lim = 1.1 * np.abs(trajectories).max()
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_zlim(-lim, lim)
ax.set_box_aspect((1, 1, 1), zoom=1.15)      # slight zoom to fill the figure
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

# Ghost of the complete orbit (one body is enough, since all three share the curve)
ax.plot(*trajectories[0].T, color="white", lw=0.6, alpha=0.12)

# Bright, saturated colours for the three bodies (one per body, as in the static plot)
colours = plt.cm.tab10(np.arange(3))

# Dynamic elements (created once, updated in every frame)
# Each trail is a Line3DCollection: a set of short segments between consecutive stored
# positions, each with its own transparency. Older segments are more transparent, so
# the trail fades out behind the body like a comet tail.
trails = []
glows = []
cores = []
arrows = []
for k in range(len(bodies)):
    # It is initialised with one degenerate segment (rather than none) because some
    # matplotlib versions fail when an empty collection is added to a 3D axes.
    trail = Line3DCollection([np.zeros((2, 3))], linewidths=2.0)
    ax.add_collection3d(trail)
    trails.append(trail)

    # Each body is drawn twice: a large, transparent "glow" and a small, solid "core"
    glow, = ax.plot([], [], [], "o", color=colours[k], markersize=18, alpha=0.25)
    core, = ax.plot([], [], [], "o", color=colours[k], markersize=7,
                    markeredgecolor="white", markeredgewidth=0.8, label=f"Body {k + 1}")
    glows.append(glow)
    cores.append(core)

    # Velocity arrow, drawn as a line from the body to r + v_scale * v
    arrow, = ax.plot([], [], [], color=colours[k], lw=2.0, alpha=0.9)
    arrows.append(arrow)

# Centre of mass marker
cm_marker, = ax.plot([], [], [], "x", color="white", markersize=8, label="Centre of mass")

# Heads-up display (2D text anchored to the figure, so it does not rotate with the camera)
hud = ax.text2D(0.02, 0.97, "", transform=ax.transAxes, color="0.9",
                family="monospace", fontsize=9, va="top")
ax.set_title("Three-body figure-eight (Velocity Verlet 4th order)", color="0.9")
legend = ax.legend(loc="upper right", fontsize=8, facecolor=bg, edgecolor="0.3", labelcolor="0.9")


# Frame update
# update(i) is called by FuncAnimation for animation frame i. It maps the animation
# frame to a stored snapshot f, and from that to the integration step and the time.
def update(i):
    f = frame_ids[i]        # index of the stored snapshot shown in this frame
    step = f * steps_per_frame      # integration step reached at this snapshot
    t = f * dt_frame        # simulated time at this snapshot

    for k in range(len(bodies)):
        r = trajectories[k, f]      # current position of body k

        # Trail: the last trail_len stored positions, turned into consecutive segments
        start = max(0, f - trail_len)
        pts = trajectories[k, start:f + 1]
        segments = np.stack([pts[:-1], pts[1:]], axis=1)        # shape: (n_segments, 2, 3)
        trails[k].set_segments(segments)

        # Transparency rises linearly from 0 (oldest segment) to 1 (newest segment)
        rgba = np.tile(colours[k], (len(segments), 1))
        rgba[:, 3] = np.linspace(0.0, 1.0, len(segments)) if len(segments) else []
        trails[k].set_color(rgba)

        # Markers: set_data_3d expects sequences, hence the single-element lists
        glows[k].set_data_3d([r[0]], [r[1]], [r[2]])
        cores[k].set_data_3d([r[0]], [r[1]], [r[2]])

        # Velocity arrow from r to r + v_scale * v
        tip = r + v_scale * velocities[k, f]
        arrows[k].set_data_3d([r[0], tip[0]], [r[1], tip[1]], [r[2], tip[2]])

    # Centre of mass (should remain on the origin throughout)
    cm = r_cm[f]
    cm_marker.set_data_3d([cm[0]], [cm[1]], [cm[2]])

    # Camera: one full turn of azimuth over the whole animation, with a gentle rise and
    # fall of the elevation, so the scene is seen from every side (and edge-on to the
    # orbital plane at the lowest elevation, which shows that the motion is planar).
    progress = i / max(1, len(frame_ids) - 1)
    ax.view_init(elev=30 + 20 * np.sin(2 * np.pi * progress), azim=-60 + 360 * progress)

    # Read-out: integration progress, time, orbital phase and the speed of each body.
    # Body 2 is fastest whenever it passes through the centre of the eight.
    lines = [
        f"step  {step:>6d} / {n_steps}",
        f"t     {t:8.4f}",
        f"t/T   {t / T_period:8.4f}",
        f"|r_cm| {np.linalg.norm(cm):.1e}",
    ]
    lines += [f"|v{k + 1}|  {speeds[k, f]:8.4f}" for k in range(len(bodies))]
    hud.set_text("\n".join(lines))

    return trails + glows + cores + arrows + [cm_marker, hud]


# Build and display the player
# FuncAnimation calls update() for every animation frame. to_jshtml() renders all
# frames to images and wraps them in a self-contained HTML/JavaScript player, which the
# notebook displays below the cell. Building it takes a little while (every frame is
# rendered once). plt.close(fig) prevents a duplicate static figure from appearing.
anim = animation.FuncAnimation(fig, update, frames=len(frame_ids), interval=1000 / fps, blit=False)
plt.close(fig)
HTML(anim.to_jshtml(fps=fps, default_mode="once"))